In [2]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("NumPy:", np.__version__)
print("FAISS:", faiss.__version__)
print("Everything is working!")

NumPy: 2.5.2
FAISS: 1.15.0
Everything is working!


In [3]:
pip install sentence_transformers 


Note: you may need to restart the kernel to use updated packages.


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!


In [5]:
documents = [
    "To reset your password, go to the login page and click on Forgot Password.",
    "You can update your account email address from the account settings page.",
    "If you forgot your username, use the account recovery option to find it.",
    "To update your billing information, open the billing section in your account.",
    "You can view your previous invoices from the billing history page.",
    "Refund requests can be submitted through the payments and billing section.",
    "If you cannot log in, check that your username and password are correct.",
    "Your account can be permanently deleted from the account settings page.",
    "Two-factor authentication can be enabled from your account security settings.",
    "If your payment failed, check your card details or try another payment method."
]

print("Number of documents:", len(documents))

Number of documents: 10


In [6]:
embeddings = model.encode(documents)

print("Embedding shape:", embeddings.shape)

Embedding shape: (10, 384)


In [7]:
# Make a copy so that we don't accidentally modify
# our original embeddings
normalized_embeddings = embeddings.copy()

# Convert to float32 because FAISS works with float32 vectors
normalized_embeddings = normalized_embeddings.astype("float32")
# Normalize every embedding to have length 1
faiss.normalize_L2(normalized_embeddings)

print("Embeddings normalized successfully!")

Embeddings normalized successfully!


In [8]:
# Create a FAISS index using L2 distance
# 384 is the number of dimensions in our embeddings
index = faiss.IndexFlatL2(384)

print("FAISS index created successfully!")

FAISS index created successfully!


In [9]:
# Add all normalized embeddings to the FAISS index
index.add(normalized_embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 10


In [10]:
def semantic_search(query, k=3):
    # Convert the user's query into an embedding
    query_embedding = model.encode([query])

    # Convert to float32 because FAISS expects float32
    query_embedding = query_embedding.astype("float32")

    # Normalize the query embedding
    faiss.normalize_L2(query_embedding)

    # Search FAISS for the top k most similar documents
    distances, indices = index.search(query_embedding, k)

    print(f"\nQuery: {query}")
    print("-" * 100)
    print(f"{'Rank':<8}{'Score':<15}Matched Sentence")
    print("-" * 100)

    # Display the results
    for rank, (distance, idx) in enumerate(
        zip(distances[0], indices[0]), start=1
    ):
        # Convert squared L2 distance to cosine similarity
        cosine_similarity = 1 - (distance / 2)

        print(
            f"{rank:<8}"
            f"{cosine_similarity:<15.4f}"
            f"{documents[idx]}"
        )

In [11]:
semantic_search("I forgot my password")


Query: I forgot my password
----------------------------------------------------------------------------------------------------
Rank    Score          Matched Sentence
----------------------------------------------------------------------------------------------------
1       0.7578         To reset your password, go to the login page and click on Forgot Password.
2       0.6194         If you forgot your username, use the account recovery option to find it.
3       0.4830         If you cannot log in, check that your username and password are correct.


In [12]:
semantic_search("I want to get my money back")


Query: I want to get my money back
----------------------------------------------------------------------------------------------------
Rank    Score          Matched Sentence
----------------------------------------------------------------------------------------------------
1       0.5802         Refund requests can be submitted through the payments and billing section.
2       0.4296         If your payment failed, check your card details or try another payment method.
3       0.3030         To reset your password, go to the login page and click on Forgot Password.


In [13]:
semantic_search("My card payment is not working")


Query: My card payment is not working
----------------------------------------------------------------------------------------------------
Rank    Score          Matched Sentence
----------------------------------------------------------------------------------------------------
1       0.7353         If your payment failed, check your card details or try another payment method.
2       0.3433         To update your billing information, open the billing section in your account.
3       0.2761         If you cannot log in, check that your username and password are correct.


In [15]:
print("======================================")
print("      SEMANTIC SEARCH ENGINE")
print("======================================")
print("Type 'exit' to quit.")

while True:

    query = input("\nEnter your query: ")

    if query.lower() == "exit":
        print("Search engine closed.")
        break

    semantic_search(query, k=3)

      SEMANTIC SEARCH ENGINE
Type 'exit' to quit.



Enter your query:  How can I reset my password?



Query: How can I reset my password?
----------------------------------------------------------------------------------------------------
Rank    Score          Matched Sentence
----------------------------------------------------------------------------------------------------
1       0.8185         To reset your password, go to the login page and click on Forgot Password.
2       0.5352         If you forgot your username, use the account recovery option to find it.
3       0.4397         If you cannot log in, check that your username and password are correct.



Enter your query:  exit


Search engine closed.


In [16]:
%%writefile theory_answers.md

# FAISS and Semantic Search — Theory Answers

## Q1. What is the difference between IndexFlatL2 and IndexFlatIP in FAISS? When would you use each?

### IndexFlatL2

`IndexFlatL2` uses L2 (Euclidean) distance to compare vectors.

A smaller L2 distance means that two vectors are more similar.

It performs an exact nearest-neighbour search by comparing the query with the vectors stored in the index.

It can be used when Euclidean distance is the required similarity measure.

### IndexFlatIP

`IndexFlatIP` uses Inner Product (dot product) to compare vectors.

A larger inner product means greater similarity.

When embeddings are normalized to unit length, inner product is equivalent to cosine similarity.

Therefore, `IndexFlatIP` is commonly used when we want cosine-similarity-based retrieval with normalized embeddings.

### Comparison

| Index | Similarity Measure | Better Result |
|---|---|---|
| `IndexFlatL2` | L2 / Euclidean distance | Lower |
| `IndexFlatIP` | Inner Product | Higher |

---

## Q2. Why do we normalise embeddings before adding them to FAISS when we want cosine similarity?

Cosine similarity measures the angle between two vectors rather than their magnitude.

The formula is:

`Cosine Similarity = (A · B) / (||A|| × ||B||)`

When vectors are normalized, their magnitude becomes 1.

Therefore, the cosine similarity can be represented using the inner product of the normalized vectors.

In this assignment, we use normalized embeddings with `IndexFlatL2`. For unit-normalized vectors, L2 distance and cosine similarity produce the same ranking of vectors.

Therefore, normalization allows us to perform similarity search while reducing the effect of vector magnitude.

---

## Q3. FAISS uses ANN (Approximate Nearest Neighbour) search. What does "approximate" mean here and why is it acceptable?

Approximate Nearest Neighbour (ANN) search means that a search algorithm may return highly similar vectors without guaranteeing that they are the mathematically exact nearest neighbours.

This approach is useful for very large datasets because exact search can become expensive when millions or billions of vectors are stored.

ANN methods trade a small amount of retrieval accuracy for significantly faster search.

For applications such as semantic search and RAG, retrieving highly relevant documents quickly is often more useful than spending significantly more time finding the mathematically exact nearest neighbour.

### Important Note

`IndexFlatL2` itself performs exact nearest-neighbour search. It is not an approximate index.

FAISS also provides approximate indexing methods such as IVF and HNSW that are designed for large-scale approximate nearest-neighbour search.

Writing theory_answers.md


In [17]:
%%writefile README.md

# Mini Semantic Search Engine using FAISS

## Overview

This project implements a mini semantic search engine using Python, Sentence Transformers, and FAISS.

The system converts knowledge-base sentences and user queries into numerical embeddings and uses FAISS to retrieve the most semantically similar sentences.

## Technologies Used

- Python
- NumPy
- Sentence Transformers
- FAISS
- all-MiniLM-L6-v2

## Project Architecture

Knowledge Base
        ↓
Embedding Model
        ↓
Document Embeddings
        ↓
Normalization
        ↓
FAISS Index
        ↓
User Query
        ↓
Query Embedding
        ↓
Normalization
        ↓
Similarity Search
        ↓
Top 3 Results

## Embedding Model

The project uses the `all-MiniLM-L6-v2` Sentence Transformer model.

Each sentence is converted into a 384-dimensional embedding.

With 10 knowledge-base sentences, the embedding matrix has the shape:

`(10, 384)`

## FAISS Index

The project uses:

`faiss.IndexFlatL2(384)`

The document embeddings are normalized before being added to the index.

The query embedding is also normalized before performing the search.

## Knowledge Base

The knowledge base contains customer-support information related to:

- Password reset
- Login issues
- Account management
- Billing
- Invoices
- Refunds
- Two-factor authentication
- Payment failures

## Features

- Generate text embeddings
- Normalize embeddings
- Store vectors using FAISS
- Perform semantic similarity search
- Retrieve Top 3 results
- Interactive CLI
- Exit using `exit`

## Example Queries

- I forgot my password
- How can I get my money back?
- My card payment is not working
- I want to change my email address
- Where can I find my invoices?

## Assignment Tasks Completed

- [x] Generate embeddings
- [x] Verify embedding shape
- [x] Normalize embeddings
- [x] Create FAISS index
- [x] Add vectors to FAISS
- [x] Perform Top 3 semantic search
- [x] Test multiple queries
- [x] Build interactive CLI
- [x] Answer theory questions

Writing README.md
